In [ ]:
from torch.nn import *
from sklearn.datasets import make_classification
import numpy as np
import torch

torch.set_default_tensor_type(torch.cuda.FloatTensor)


class BatchNorm(BatchNorm1d):
    def forward(s, x):
        return super().forward(x.view(-1, x.size(-1))).view(x.shape)


def los(x, y):
    return functional.binary_cross_entropy_with_logits(x, y.broadcast_to(x.shape))


def acc(x, y):
    return (x.signbit() != y).sum() / x.numel()


def dat(n, l, b=1):
    return torch.tensor(
        np.stack(
            [
                np.concatenate(
                    (
                        a[0].astype("f"),
                        np.unpackbits(a[1].view("B")[::4, None], 1)[:, -n:].astype("f"),
                    ),
                    1,
                )
                for a in range(b)
                for a in [
                    make_classification(
                        l,
                        n,
                        n_informative=n,
                        n_redundant=0,
                        n_classes=2 ** n,
                        n_clusters_per_class=1,
                    )
                ]
            ],
            1,
        )
    )


def train(M, a):
    return M(a)[0]


def fast_eval(m, d, x, y):
    i = torch.randint(len(d), d.shape[:1]) % torch.arange(1, len(d) + 1)
    return torch.cat((d, x[i]), 2), y[i]


def slow_eval(m, d, x, y):
    i, I = torch.tril_indices(*d.shape[:1] * 2)
    return m(torch.cat((d[i], x[I]), 2)), y[I]


def full_eval(m, d, x, y):
    return (
        m(
            torch.cat(
                (
                    d[:, None].broadcast_to(len(d), *d.shape),
                    x.broadcast_to(len(d), *x.shape),
                ),
                3,
            )
        ),
        y,
    )

In [ ]:
n = 7
m = 3 * 2 ** n
M = GRU(n * 2, m)
m = Sequential(
    Linear(m + n, m, 0),
    BatchNorm(m, track_running_stats=0),
    Hardswish(1),
    Linear(m, n),
)
o = torch.optim.Adam((*M.parameters(), *m.parameters()))
s = torch.cuda.amp.GradScaler()
sum(i.numel() for i in o.param_groups[0]["params"])

In [ ]:
while 1:
    for i in range(2000):
        o.zero_grad()
        with torch.cuda.amp.autocast():
            a = dat(n, 64 * 12)
            a = slow_eval(m, train(M, a), a[..., :n], a[..., n:])
            i = los(*a)
        s.scale(i).backward()
        s.step(o)
        s.update()
    print(i, acc(*a))

torch.save((M,m,o),'nnTrainerEvaler-Copy2.model.pk')